In [26]:
!pip install -q transformers datasets accelerate

In [27]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [28]:
print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Training will use CPU.")

PyTorch version: 2.11.0+cu128
GPU available: Tesla T4


In [30]:
# Step 4: Create a custom text dataset

texts = [
    "Artificial intelligence is transforming the modern world by helping people solve complex problems and automate repetitive tasks.",

    "Machine learning allows computers to learn patterns from data and make useful predictions without being explicitly programmed for every task.",

    "Generative AI can create text, images, code and other forms of content based on instructions provided by a user.",

    "Technology has changed education by providing students with online courses, digital libraries and interactive learning platforms.",

    "Python is one of the most popular programming languages for artificial intelligence and machine learning because of its simple syntax and powerful libraries.",

    "Data is an important resource for artificial intelligence systems because machine learning models learn patterns from large collections of examples.",

    "Deep learning uses neural networks with multiple layers to learn complex patterns from data.",

    "Natural language processing allows computers to understand, process and generate human language.",

    "Artificial intelligence is being used in healthcare to assist doctors, analyze medical information and improve patient care.",

    "Cybersecurity systems can use machine learning to identify unusual patterns and detect potential threats.",

    "Cloud computing provides access to computing resources over the internet and is widely used for modern software applications.",

    "Software developers use programming languages and frameworks to build applications that solve real-world problems.",

    "The future of artificial intelligence will depend on responsible development, reliable data and careful evaluation of AI systems.",

    "Robotics combines software, hardware and artificial intelligence to create machines that can perform useful tasks.",

    "Computer vision enables machines to analyze and understand information from images and videos.",

    "AI assistants can help users find information, generate content and complete everyday tasks more efficiently.",

    "Learning programming improves problem-solving skills because programmers need to break large problems into smaller logical steps.",

    "Large language models are trained on text data and can generate human-like responses based on patterns learned during training.",

    "A good machine learning model requires suitable data, appropriate training methods and proper evaluation.",

    "Artificial intelligence is becoming an important part of many industries including finance, education, healthcare and entertainment."
]

print("Number of training examples:", len(texts))
print("\nFirst example:")
print(texts[0])

Number of training examples: 20

First example:
Artificial intelligence is transforming the modern world by helping people solve complex problems and automate repetitive tasks.


In [31]:
# Step 5: Convert our text data into a Hugging Face Dataset

from datasets import Dataset

# Create Hugging Face dataset
dataset = Dataset.from_dict({
    "text": texts
})

# Display dataset information
print("Dataset created successfully!")
print(dataset)

# Show first 3 examples
print("\nFirst 3 training examples:")
for i in range(3):
    print(f"\nExample {i+1}:")
    print(dataset[i]["text"])

Dataset created successfully!
Dataset({
    features: ['text'],
    num_rows: 20
})

First 3 training examples:

Example 1:
Artificial intelligence is transforming the modern world by helping people solve complex problems and automate repetitive tasks.

Example 2:
Machine learning allows computers to learn patterns from data and make useful predictions without being explicitly programmed for every task.

Example 3:
Generative AI can create text, images, code and other forms of content based on instructions provided by a user.


In [32]:
# Step 6: Load GPT-2 tokenizer

from transformers import AutoTokenizer

# Load GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# GPT-2 does not have a padding token by default
tokenizer.pad_token = tokenizer.eos_token

print("GPT-2 tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)
print("Padding token:", tokenizer.pad_token)

GPT-2 tokenizer loaded successfully!
Vocabulary size: 50257
Padding token: <|endoftext|>


In [33]:
# Step 7: Tokenize the custom dataset

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("Dataset tokenized successfully!")
print(tokenized_dataset)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset tokenized successfully!
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 20
})


In [34]:
# Step 8: Load the GPT-2 model

from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("gpt2")

# Set padding token
model.config.pad_token_id = tokenizer.pad_token_id

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("GPT-2 model loaded successfully!")
print("Device:", device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 model loaded successfully!
Device: cuda


In [35]:
# Step 9: Prepare the data collator

from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator created successfully!")

Data collator created successfully!


In [36]:
# Step 10: Set up training arguments

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-text-generation",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
    fp16=True
)

print("Training arguments configured successfully!")

Training arguments configured successfully!


In [37]:
# Step 11: Training setup

training_args = TrainingArguments(
    output_dir="./gpt2-text-generation",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=10,
    learning_rate=5e-5,
    report_to="none"
)

print("Training arguments created successfully!")


Training arguments created successfully!


In [38]:
# Step 12: Create Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Trainer created successfully!")

Trainer created successfully!


In [39]:
# Step 13: Fine-tune GPT-2

print("Starting GPT-2 fine-tuning...")

trainer.train()

print("GPT-2 fine-tuning completed successfully!")

Starting GPT-2 fine-tuning...


Step,Training Loss
10,3.258878
20,2.313546
30,1.974438


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

GPT-2 fine-tuning completed successfully!


In [40]:
# Step 14: Generate text using the fine-tuned GPT-2 model

from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

prompt = "Artificial intelligence is"

result = generator(
    prompt,
    max_length=80,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.8,
    top_k=50
)

print("Generated Text:")
print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated Text:
Artificial intelligence is being used to build artificial intelligence systems such as artificial intelligence and machine learning. The technology is designed to develop machines that perform tasks such as learning patterns and predicting future events.

The company's researchers developed prototypes that can perform tasks such as finding coupons and shopping online. One of the first computers built on the IBM Watson machine quickly learns patterns from a customer's photos and generates patterns from them. The company began developing artificial intelligence systems for entertainment, educational programs and medical diagnostics. The company also is developing artificial intelligence and artificial intelligence systems for medical and entertainment applications.

According to IBM, IBM has more than 100 million employees worldwide and employs more than 100 million. IBM employs more than $20 billion in employees worldwide.

The U.S. government invests $5 billion a year t